# 01 — Introduction (CSS)

A guided tour of the CSS builder (level 1). Each section is a
small self-contained example showing the code, the rendered CSS
source, and a live HTML demo.

In this dialect a **selector is the top-level container** of a
case: it holds the rule (properties), media/supports variants,
nested selectors. When the same block applies to multiple
selectors, wrap them in a `selector_list`.

**Topics:**
1. Single selector + rule.
2. Selector list — multiple selectors share one block.
3. Selectors built from kwargs.
4. Media variants inheriting the parent selector.
5. CSS Nesting via nested selectors.
6. CSS variables and comments.

## 1. Single selector + rule

A selector at the top, a rule (the property block) attached
to it. The rule is what the browser will apply when the
selector matches.

In [ ]:
from IPython.display import HTML, Code

from genro_builders.contrib.css import CssBuilderHandler


class HelloCss(CssBuilderHandler):
    def main(self, root):
        s = root.selector(_class="card")
        s.rule(color="white", background_color="#3498db", padding="12px")


page = HelloCss()
page.create()
page.build()
css = page.render()

In [ ]:
Code(css, language='css')

In [ ]:
HTML(f"<style>{css}</style><div class='card'>Hello CSS</div>")

## 2. Selector list — multiple selectors share one block

`selector_list` is the explicit container for a comma-separated
selector-list. Add N `selector` children, then attach the
shared rule and variants once.

In [ ]:
class Shared(CssBuilderHandler):
    def main(self, root):
        sl = root.selector_list()
        sl.selector(_class="card")
        sl.selector(_class="panel")
        sl.selector(_class="dialog")
        sl.rule(padding="8px", border_radius="4px",
                background_color="#3498db", color="white")


page = Shared()
page.create()
page.build()
css = page.render()

In [ ]:
Code(css, language='css')

In [ ]:
HTML(
    f"<style>{css}</style>"
    "<span class='card'>Card</span> "
    "<span class='panel'>Panel</span> "
    "<span class='dialog'>Dialog</span>"
)

## 3. Selectors built from kwargs

`selector(...)` accepts structured kwargs (`tag`, `id`,
`_class`, `classes`, `attr`) and an opaque `raw` suffix.
Validation is eager: malformed values raise `ValueError` at
render time.

In [ ]:
class Selectors(CssBuilderHandler):
    def main(self, root):
        sheet = root.stylesheet()

        compound = sheet.selector(classes=["card", "highlighted"])
        compound.rule(border="2px solid #3498db", padding="8px")

        inp = sheet.selector(tag="input", attr={"type": "text"})
        inp.rule(background_color="#f8f8f8", padding="4px")

        hover = sheet.selector(_class="card:hover")
        hover.rule(color="#3498db", cursor="pointer")


page = Selectors()
page.create()
page.build()
css = page.render()

In [ ]:
Code(css, language='css')

In [ ]:
HTML(
    f"<style>{css}</style>"
    "<div class='card highlighted'>Compound</div>"
    "<input type='text' value='input with grey bg'/>"
)

## 4. Media variants

Inside a selector, `media(condition="...", **properties)` adds
a `@media` variant. Properties apply to the parent selector
inside the `@media` block — you don't need to repeat the
selector.

In [ ]:
class Responsive(CssBuilderHandler):
    def main(self, root):
        card = root.selector(_class="responsive-card")
        card.rule(width="300px", padding="16px")
        card.media(condition="(max-width: 600px)",
                   width="100%", padding="8px")
        card.media(condition="(max-width: 400px)",
                   padding="4px")


page = Responsive()
page.create()
page.build()
css = page.render()

In [ ]:
Code(css, language='css')

In [ ]:
HTML(
    f"<style>{css}</style>"
    "<div class='responsive-card' style='background:#3498db;color:white'>"
    "Resize the window to see the responsive behavior."
    "</div>"
)

## 5. CSS Nesting

A selector can host nested selectors. Each nested selector
becomes a nested block in the output CSS. Use `raw="&..."` to
bind tightly to the parent (e.g. `&:hover`).

In [ ]:
class Nested(CssBuilderHandler):
    def main(self, root):
        card = root.selector(_class="nested-card")
        card.rule(padding="8px", background_color="#fafafa")

        title = card.selector(_class="title")
        title.rule(font_size="18px", font_weight="bold")

        hover = card.selector(raw="&:hover")
        hover.rule(background_color="#eef")


page = Nested()
page.create()
page.build()
css = page.render()

In [ ]:
Code(css, language='css')

In [ ]:
HTML(
    f"<style>{css}</style>"
    "<div class='nested-card'>"
    "  <div class='title'>Title</div>"
    "  Body. Hover the block to see the background change."
    "</div>"
)

## 6. CSS variables and comments

`cssvar(name, value=..., comment=...)` declares a custom
property. Any element accepts `comment="..."`: short comments
(≤ 60 chars) go inline, long ones go above as a block.

In [ ]:
class Theming(CssBuilderHandler):
    def main(self, root):
        sheet = root.stylesheet()

        rt = sheet.selector(raw=":root")
        rt.cssvar("brand", value="#3498db", comment="brand color")
        rt.cssvar("spacing", value="8px")

        card = sheet.selector(_class="themed-card",
                              comment="branded card")
        card.rule(background_color="var(--brand)",
                  color="white",
                  padding="var(--spacing)")


page = Theming()
page.create()
page.build()
css = page.render()

In [ ]:
Code(css, language='css')

In [ ]:
HTML(
    f"<style>{css}</style>"
    "<div class='themed-card'>Themed via CSS variables</div>"
)